# BMC Discovery Healthcheck

Single notebook for collecting API and CLI evidence for a BMC Discovery healthcheck. The section order follows the TekWurx healthcheck template, while output remains notebook-native CSV/TXT evidence rather than a generated Word document.

## Controls

Set `WRITE_OUTPUT = True` to write canonical report artifacts under each appliance output directory. By default all appliances in `config.yaml` are selected.

In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import sys
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML(str):
        pass

    def display(value):
        print(value)

def find_repo_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "config.yaml").exists() or (candidate / ".git").is_dir():
            return candidate
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import tideway
from tideway import notebooks as tw_nb
from tideway.results import BatchReportResult, ReportResult, TextResult

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
display(HTML("""
<style>
.jp-Notebook .jp-OutputArea-output table,
.jp-Notebook .jp-RenderedHTMLCommon table,
.jp-Notebook table.dataframe,
.jp-OutputArea-output table,
.jp-RenderedHTMLCommon table,
table.dataframe {
    width: 100% !important;
}
.jp-Notebook table.dataframe th,
.jp-Notebook table.dataframe td,
table.dataframe th,
table.dataframe td {
    white-space: normal !important;
    word-break: break-word;
}
</style>
"""))

def show_table(frame):
    if frame is None or frame.empty:
        print("No rows to display.")
        return
    html = frame.to_html(border=1)
    html = html.replace('<table border="1" class="dataframe">', '<table border="1" class="dataframe" style="width:100%">', 1)
    display(HTML(html))


RUN_API = True
RUN_CLI = True
WRITE_OUTPUT = True
OUTPUT_BASE_DIR = None

INCLUDE_APPLIANCES = None
EXCLUDE_APPLIANCES = []

DISK_USED_THRESHOLD = 70
DISPLAY_ROW_LIMIT = 50
SYSTEM_DEFAULT_USERNAME = "system"

API_REPORTS = [
    "credential_success",
    "schedules",
    "excludes",
    "discovery_analysis",
    "discovery_run_analysis",
    "active_scans",
    "sensitive_data",
    "eca_errors",
    "open_ports",
    "host_utilisation",
    "orphan_vms",
    "missing_vms",
    "near_removal",
    "removed",
    "os_lifecycle",
    "software_lifecycle",
    "db_lifecycle",
    "unrecognised_snmp",
    "capture_candidates",
    "installed_agents",
    "expected_agents",
    "si_user_accounts",
    "tku",
    "vault",
    "outpost_creds",
    "licensing",
]

CLI_CHECKS = [
    "health_check",
    "baseline",
    "host_info",
    "disk_info",
    "disk_usage_alerts",
    "ntp",
    "vmware_tools",
    "dns_resolution",
    "core_dumps",
    "tw_crontab",
    "tw_config_dump",
    "tw_options",
    "clustering",
    "cmdb_errors",
    "ldap",
    "cli_users",
    "syslog",
    "certificates",
    "ds_status",
    "playback_data",
]

## Configuration

The notebook merges top-level config values with each `appliances` entry. API and CLI failures are recorded per appliance so one failing check does not stop the collection.

In [ ]:
def config_bool(value, default=False):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    return default


def appliance_entries(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    entries = config.get("appliances") or []
    if isinstance(entries, dict):
        entries = [entries]
    if not entries:
        entries = [{"name": config.get("name") or config.get("target") or "default"}]
    return entries


def merged_appliance(config: Dict[str, Any], entry: Dict[str, Any]) -> Dict[str, Any]:
    merged = dict(config)
    merged.update(entry or {})
    merged.pop("appliances", None)
    return merged


def resolve_optional_path(value, repo_root: Path):
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = repo_root / path
    return str(path)




def version_sort_key(value):
    parts = []
    for part in str(value).lstrip("v").split("."):
        try:
            parts.append(int(part))
        except ValueError:
            parts.append(0)
    return tuple(parts)


def api_version_for(entry: Dict[str, Any], config: Dict[str, Any], target: str, token: Optional[str], verify_ssl: bool):
    configured = entry.get("api_version") or config.get("api_version")
    if configured:
        return str(configured).lstrip("v"), "configured"
    try:
        response = tideway.appliance(target, token or "", ssl_verify=verify_ssl).api_about
        response.raise_for_status()
        versions = response.json().get("api_versions") or []
        if versions:
            return str(sorted(versions, key=version_sort_key)[-1]).lstrip("v"), "detected"
    except Exception as exc:
        return "1.16", f"defaulted after detection error: {exc}"
    return "1.16", "defaulted; /api/about returned no api_versions"
repo_root = tw_nb.find_repo_root()
config = tw_nb.load_config()
include_set = set(INCLUDE_APPLIANCES or [])
exclude_set = set(EXCLUDE_APPLIANCES or [])

appliances = []
config_errors = []

for raw_entry in appliance_entries(config):
    entry = merged_appliance(config, raw_entry)
    target = str(entry.get("target") or "").strip()
    name = str(entry.get("name") or target or "default").strip()
    if include_set and name not in include_set and target not in include_set:
        continue
    if name in exclude_set or target in exclude_set:
        continue
    if not target:
        config_errors.append({"appliance": name, "error": "missing target"})
        continue

    token = None
    token_error = None
    try:
        token = tw_nb.token_from_config(entry, repo_root)
    except Exception as exc:
        token_error = str(exc)

    verify_ssl = config_bool(entry.get("verify_ssl", entry.get("ssl_verify", config.get("verify_ssl", False))), default=False)
    api_version, api_version_source = api_version_for(entry, config, target, token, verify_ssl)
    output_dir = tw_nb.output_dir_for(target, OUTPUT_BASE_DIR)
    tw = tideway.appliance(target, token, api_version=api_version, ssl_verify=verify_ssl) if token else None

    cli_kwargs = {
        "username": entry.get("username") or "tideway",
        "password": entry.get("password"),
        "password_file": resolve_optional_path(entry.get("password_file") or entry.get("f_passwd"), repo_root),
        "system_username": entry.get("system_username") or SYSTEM_DEFAULT_USERNAME,
        "system_password": entry.get("system_password"),
        "system_password_file": resolve_optional_path(entry.get("system_password_file"), repo_root),
    }
    has_cli_auth = bool(cli_kwargs.get("password") or cli_kwargs.get("password_file"))

    appliances.append({
        "name": name,
        "target": target,
        "api_version": api_version,
        "api_version_source": api_version_source,
        "verify_ssl": verify_ssl,
        "token_error": token_error,
        "tw": tw,
        "output_dir": output_dir,
        "cli_kwargs": cli_kwargs,
        "has_cli_auth": has_cli_auth,
    })

inventory_df = pd.DataFrame([
    {
        "Appliance": item["name"],
        "Target": item["target"],
        "API Version": item["api_version"],
        "API Version Source": item["api_version_source"],
        "API Ready": item["tw"] is not None,
        "CLI Ready": item["has_cli_auth"],
        "Output Dir": str(item["output_dir"]),
        "API Config Error": item["token_error"],
    }
    for item in appliances
])

print(f"Repo root: {repo_root}")
show_table(inventory_df)
if config_errors:
    show_table(pd.DataFrame(config_errors))

## Evidence Helpers

In [ ]:
api_results: Dict[tuple, Any] = {}
cli_results: Dict[tuple, Any] = {}
status_rows: List[Dict[str, Any]] = []


def record_status(appliance, check, source, status, rows=None, error=None, files=None):
    status_rows.append({
        "Appliance": appliance,
        "Check": check,
        "Source": source,
        "Status": status,
        "Rows": rows,
        "Files": ", ".join(files or []),
        "Error": error,
    })


def result_rows(result):
    if isinstance(result, BatchReportResult):
        return sum(result_rows(item) or 0 for item in result.results)
    if isinstance(result, ReportResult):
        return len(result.rows)
    if isinstance(result, TextResult):
        return len(result.text.splitlines())
    return None


def response_to_frame(response):
    if response is None:
        return pd.DataFrame()
    try:
        payload = response.json()
    except Exception:
        return pd.DataFrame({"text": str(getattr(response, "text", "")).splitlines()})
    if isinstance(payload, list):
        return pd.DataFrame(payload)
    if isinstance(payload, dict):
        for key in ("results", "items", "data"):
            if isinstance(payload.get(key), list):
                return pd.DataFrame(payload[key])
        return pd.DataFrame([payload])
    return pd.DataFrame({"value": [payload]})


def result_to_frame(result):
    if isinstance(result, BatchReportResult):
        frames = []
        for item in result.results:
            frame = result_to_frame(item)
            if not frame.empty:
                frame.insert(0, "Report", item.name)
                frames.append(frame)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return tw_nb.report_to_dataframe(result)


def show_text_result(label, result, max_lines=20):
    if isinstance(result, TextResult):
        text = "\n".join(result.text.splitlines()[:max_lines])
        print(f"\n## {label}\n{text}")


def show_result_table(title, result, limit=DISPLAY_ROW_LIMIT):
    frame = result_to_frame(result)
    if frame.empty:
        show_text_result(title, result)
    else:
        print(title)
        show_table(frame.head(limit))


def status_frame(checks=None, source=None):
    frame = pd.DataFrame(status_rows)
    if frame.empty:
        return frame
    if checks is not None:
        frame = frame[frame["Check"].isin(checks)]
    if source is not None:
        frame = frame[frame["Source"] == source]
    return frame.reset_index(drop=True)


def display_status(checks):
    frame = status_frame(checks=checks)
    if frame.empty:
        print("No checks collected for this section.")
    else:
        show_table(frame)

## Direct API Checks

In [ ]:
DIRECT_API_CHECKS = {
    "admin_about": lambda tw: tw.admin().get_admin_about(),
    "admin_baseline": lambda tw: tw.admin().get_admin_baseline(),
    "admin_instance": lambda tw: tw.admin().instance(),
    "admin_cluster": lambda tw: tw.admin().cluster(),
    "admin_preferences": lambda tw: tw.admin().preferences(),
    "admin_smtp": lambda tw: tw.admin().smtp(),
    "discovery_status": lambda tw: tw.discovery().getDiscoveryStatus(),
    "discovery_outposts": lambda tw: tw.discovery().get_discovery_outpost(),
}

if RUN_API:
    for item in appliances:
        tw = item["tw"]
        if tw is None:
            record_status(item["name"], "api_config", "api", "skipped", error=item["token_error"])
            continue
        for check_name, check_func in DIRECT_API_CHECKS.items():
            try:
                response = check_func(tw)
                api_results[(item["name"], check_name)] = response
                ok = bool(getattr(response, "ok", False))
                frame = response_to_frame(response)
                status = "ok" if ok else f"http {getattr(response, 'status_code', 'unknown')}"
                record_status(item["name"], check_name, "api", status, rows=len(frame))
            except Exception as exc:
                record_status(item["name"], check_name, "api", "error", error=str(exc))

show_table(status_frame(source="api"))

## Report API Checks

In [ ]:
if RUN_API:
    for item in appliances:
        tw = item["tw"]
        if tw is None:
            continue
        reports = tw.reports()
        output_dir = str(item["output_dir"]) if WRITE_OUTPUT else None
        for report_name in API_REPORTS:
            try:
                method = getattr(reports, report_name)
                result = method(output_dir=output_dir) if output_dir else method()
                api_results[(item["name"], report_name)] = result
                record_status(
                    item["name"],
                    report_name,
                    "report_api",
                    "ok",
                    rows=result_rows(result),
                    files=getattr(result, "files", []),
                )
            except Exception as exc:
                record_status(item["name"], report_name, "report_api", "error", error=str(exc))

show_table(status_frame(source="report_api"))

## CLI Checks

In [ ]:
def run_cli_check(cli, check_name, output_dir):
    kwargs = {"output_dir": output_dir} if output_dir else {}
    if check_name == "disk_usage_alerts":
        return cli.disk_usage_alerts(threshold=DISK_USED_THRESHOLD, **kwargs)
    return getattr(cli, check_name)(**kwargs)


if RUN_CLI:
    for item in appliances:
        if not item["has_cli_auth"]:
            for check_name in CLI_CHECKS:
                record_status(item["name"], check_name, "cli", "skipped", error="missing SSH password or password_file")
            continue

        output_dir = str(item["output_dir"]) if WRITE_OUTPUT else None
        try:
            with tideway.appliance_cli(item["target"], **item["cli_kwargs"]) as cli:
                for check_name in CLI_CHECKS:
                    if check_name == "tw_options" and not (
                        item["cli_kwargs"].get("system_password") or item["cli_kwargs"].get("system_password_file")
                    ):
                        record_status(item["name"], check_name, "cli", "skipped", error="missing system password")
                        continue
                    try:
                        result = run_cli_check(cli, check_name, output_dir)
                        cli_results[(item["name"], check_name)] = result
                        record_status(
                            item["name"],
                            check_name,
                            "cli",
                            "ok",
                            rows=result_rows(result),
                            files=getattr(result, "files", []),
                        )
                    except Exception as exc:
                        record_status(item["name"], check_name, "cli", "error", error=str(exc))
        except Exception as exc:
            for check_name in CLI_CHECKS:
                if (item["name"], check_name) not in cli_results:
                    record_status(item["name"], check_name, "cli", "error", error=str(exc))

show_table(status_frame(source="cli"))

## Baseline Checks

In [ ]:
BASELINE_CHECKS = [
    "health_check",
    "admin_baseline",
    "baseline",
    "admin_about",
    "host_info",
    "disk_info",
    "disk_usage_alerts",
    "ntp",
    "vmware_tools",
    "dns_resolution",
    "core_dumps",
    "tw_crontab",
    "tw_config_dump",
    "tw_options",
    "admin_cluster",
    "clustering",
    "cmdb_errors",
    "discovery_outposts",
    "licensing",
]
display_status(BASELINE_CHECKS)

for key, result in cli_results.items():
    if key[1] in {"disk_usage_alerts", "disk_info"}:
        show_result_table(f"{key[0]} - {key[1]}", result)


## Security Checks

In [ ]:
SECURITY_CHECKS = [
    "admin_about",
    "ldap",
    "cli_users",
    "vault",
    "syslog",
    "certificates",
    "admin_smtp",
    "sensitive_data",
    "si_user_accounts",
]
display_status(SECURITY_CHECKS)

for key, result in cli_results.items():
    if key[1] in {"ldap", "cli_users", "syslog", "certificates"}:
        show_result_table(f"{key[0]} - {key[1]}", result)


## Model Checks

In [ ]:
MODEL_CHECKS = [
    "tku",
    "discovery_status",
    "credential_success",
    "outpost_creds",
    "unrecognised_snmp",
    "capture_candidates",
    "installed_agents",
    "expected_agents",
    "eca_errors",
    "ds_status",
    "playback_data",
]
display_status(MODEL_CHECKS)

for key, result in api_results.items():
    if key[1] in {"tku", "credential_success", "outpost_creds", "unrecognised_snmp"}:
        show_result_table(f"{key[0]} - {key[1]}", result)


## Insights

In [ ]:
INSIGHT_CHECKS = [
    "discovery_analysis",
    "discovery_run_analysis",
    "active_scans",
    "schedules",
    "excludes",
    "sensitive_data",
    "eca_errors",
    "host_utilisation",
    "near_removal",
    "removed",
    "open_ports",
    "orphan_vms",
    "missing_vms",
    "os_lifecycle",
    "software_lifecycle",
    "db_lifecycle",
    "licensing",
]
display_status(INSIGHT_CHECKS)

for key, result in api_results.items():
    if key[1] in {"discovery_analysis", "discovery_run_analysis", "sensitive_data", "eca_errors", "host_utilisation", "near_removal", "open_ports", "orphan_vms", "os_lifecycle"}:
        show_result_table(f"{key[0]} - {key[1]}", result)


## Collection Summary

In [ ]:
summary = status_frame()
show_table(summary)

if not summary.empty:
    grouped_summary = (
        summary.groupby(["Appliance", "Source", "Status"], dropna=False)
        .size()
        .reset_index(name="Count")
        .sort_values(["Appliance", "Source", "Status"])
    )
    show_table(grouped_summary)

if WRITE_OUTPUT:
    for item in appliances:
        print(f"{item['name']}: {item['output_dir']}")